Copyright Matlantis Corp. as contributors to Matlantis contrib project

# アンブレラサンプリングの実行

アンブレラサンプリングでは、系に対して目的の場所に留まるよう、「アンブレラポテンシャル」による束縛をかけることで、エネルギーが高い不安定な状態（遷移状態など）など通常の分子シミュレーションではレアイベントとなる構造もサンプリングすることができる手法となります。

ここまでの工程では反応座標（Cu原子のz座標）に沿った初期構造を用意しました。
このノートブックでは、それぞれの初期構造に対して、アンブレラポテンシャルを課したMDシミュレーションを `joblib` で並列実行していきます。

## Step 1. ライブラリのインポートと環境設定

まずは必要なライブラリを読み込み、PLUMEDをPythonから利用するための環境変数を設定します。

In [ ]:
# 必要に応じてパッケージをインストールしてください
#!pip install joblib
#!pip install plumed

In [ ]:
# ============================================================
# 1. Environment & Imports
# ============================================================

import os
import sys
import numpy as np
from time import perf_counter
from joblib import Parallel, delayed

# ASE
from ase import units
from ase.io import read, write
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
from ase.constraints import FixAtoms,Atoms

# PLUMED wrapper
from ase.calculators.plumed import Plumed

# PFP (Matlantis)
from pfp_api_client.pfp.calculators.ase_calculator import ASECalculator
from pfp_api_client.pfp.estimator import Estimator

# PLUMED Environment Variables (Please modify if necessary to match the path of your environment.)
plumed_path = "/home/jovyan/local/plumed-2.9.0"
os.environ["PLUMED_KERNEL"] = f"{plumed_path}/lib/libplumedKernel.so"
os.environ["PLUMED_TYPESAFE_IGNORE"] = "yes"
sys.path.append(plumed_path)

## Step 2. 計算パラメータの設定

シミュレーションの温度、時間、およびアンブレラサンプリングを行う反応座標の範囲を設定します。
また、`joblib` による並列数もここで指定します。

* **N_JOBS**: 同時に走らせる計算の数です。
* **colvars**: アンブレラサンプリングを行うターゲット距離のリストです（例: 13.0Å, 13.5Å, ...）。

In [ ]:
# ============================================================
# 2. Parameters & Settings
# ============================================================

# PFP Settings
CALC_MODE     = "PBE_PLUS_D3"
METHOD_TYPE   = "PFVM"
MODEL_VERSION = "v8.0.0"

# Joblib Settings (並列計算の設定)
N_JOBS  = 10           # 同時に実行する数 (環境に合わせて調整してください)
VERBOSE = 10           # 進捗表示の詳細度
BACKEND = "threading"  # Matlantis利用時は "threading" 推奨

# MD Settings
TEMPERATURE  = 375.0        # Kelvin
TIMESTEP     = 1.0 * units.fs
TOTAL_STEPS  = 50_000       # 各ウィンドウのステップ数
LOG_INTERVAL = 100

# Reaction Coordinate (Umbrella Windows)
# 10.2Å から 18.0Å まで 0.2Å 刻み (40ウィンドウ)
cv_restraint = np.linspace(10.2, 18.0, 40)

print(f"Target Windows: {len(cv_restraint)}")
print(f"Values: {cv_restraint}")

## Step 3. 各ウィンドウについてアンブレラサンプリングを実行する関数の定義
並列化するために、「ある1つの拘束位置（cv_at）を受け取って、MD計算を行い、ファイルを保存する」という一連の処理を関数（run_us_window）にまとめます。

#### 注意点
- `Estimator` (PFPの計算機) はこの関数の中で生成します（並列化の際のトラブルを防ぐため）。
- Cu原子の下層固定（`FixAtoms`）の設定もここで行います。
- PLUMEDの設定文字列を、受け取った `z_at` に応じて動的に作成します。

#### 設定項目について
| 項目 | 説明 |
|:-----|:-----|
|`UNITS`| 単位系の設定。ここでは長さは Å、エネルギーは eV に指定します。|
|`POSITION`| 集団変数として原子の位置を設定します。<br><br> **(注意事項)**: PLUMEDの原子インデックスは**1始まり**です。ASE（Python）の**0始まり**とズレがあるため、指定する際は `ASEのインデックス + 1` の値を設定してください。 |
|`RESTRAINT` | 調和振動子型の拘束をかけます。 <br><br> - `ARG`: 対象となる集団変数を引数として指定します。<br> - `KAPPA`: 調和振動子のバネ定数を指定します。 <br> - `AT`: 調和振動子を仕掛ける場所を指定します。|
|`PRINT` | ログの出力設定。CVの値やバイアス量を `COLVAR` ファイルに出力します。|


In [ ]:
# ============================================================
# 3. Parallel Function Definition
# ============================================================
def run_us_window(cv_at):
    """
    指定された反応座標 z_at (Z方向の距離) でアンブレラサンプリングを実行する関数
    """
    s_time = perf_counter()
    
    # 文字列変換 (ファイル名やPLUMED入力用)
    cv_at_str = f"{cv_at:.2f}"
    
    # --------------------------------------------------------
    # 1. Prepare Directories
    # --------------------------------------------------------
    out_dir = f"./output/04_umbrella_sampling/cv_at_{cv_at_str}"
    os.makedirs(out_dir, exist_ok=True)

    # --------------------------------------------------------
    # 2. Prepare Calculator
    # --------------------------------------------------------
    estimator = Estimator(
        calc_mode=CALC_MODE,
        method_type=METHOD_TYPE,
        model_version=MODEL_VERSION
    )
    calculator = ASECalculator(estimator)

    # --------------------------------------------------------
    # 3. Prepare Atoms & Constraints
    # --------------------------------------------------------
    # inputsから読み込み、なければassetsから読み込む
    input_xyz = f'./inputs/initial_rc_{cv_at_str}.xyz'
    if not os.path.exists(input_xyz):
        input_xyz = f'./assets/05_umbrella_sampling/initial_rc_{cv_at_str}.xyz'
    
    if not os.path.exists(input_xyz):
        return f"Error: Input file not found for cv_at={cv_at_str}"

    atoms = read(input_xyz)
    
    # Cuの1, 2層目を固定する (z座標が閾値以下のものを固定)
    thresh = 4.0
    constraint = FixAtoms(mask=atoms.positions[:, 2] < thresh)
    atoms.set_constraint(constraint)

    # --------------------------------------------------------
    # 4. PLUMED Settings
    # --------------------------------------------------------
    # アンブレラポテンシャルの設定
    # KAPPA=2.5 eV (約 57.6 kcal/mol) で z_at の位置に拘束
    plumed_setting = [
        f"UNITS LENGTH=A ENERGY=eV",
        
        # 1. 束縛対象の原子座標を取得 (PLUMEDは1始まりのindexを使用することに注意)
        f"pos: POSITION ATOM=268",

        # 2. アンブレラポテンシャル (Harmonic Restraint)
        f"restraint-z: RESTRAINT ARG=pos.z KAPPA=2.5 AT={cv_at_str}",

        # 3. 出力設定
        f"PRINT STRIDE={LOG_INTERVAL} ARG=pos.z,restraint-z.bias,restraint-z.force2 FILE={out_dir}/COLVAR_{cv_at_str}",
        "FLUSH STRIDE=1000"
    ]
    
    # PLUMED Calculatorを接続
    atoms.calc = Plumed(
        calc=calculator,
        input=plumed_setting,
        timestep=TIMESTEP,
        atoms=atoms,
        kT=units.kB * TEMPERATURE
    )

    # --------------------------------------------------------
    # 5. MD Simulation Setup
    # --------------------------------------------------------
    # 初速度の設定
    MaxwellBoltzmannDistribution(atoms, temperature_K=TEMPERATURE, force_temp=True)
    Stationary(atoms)

    # Langevin Dynamicsの設定
    dyn = Langevin(
        atoms, 
        TIMESTEP, 
        temperature_K=TEMPERATURE, 
        friction=0.002/units.fs, 
        trajectory=f'{out_dir}/md-dyn.traj', 
        logfile=f'{out_dir}/md-dyn.log', 
        loginterval=LOG_INTERVAL
    )

    # --------------------------------------------------------
    # 6. Run Execution
    # --------------------------------------------------------
    try:
        dyn.run(TOTAL_STEPS)
        
        # Restartファイルの保存
        write(f'{out_dir}/md-dyn-restart.cif', atoms)
        write(f'{out_dir}/md-dyn-restart.xyz', atoms)
        
        elapsed_time = perf_counter() - s_time
        return f"Done: z_at={cv_at_str} ({elapsed_time:.1f} sec)"
        
    except Exception as e:
        return f"Failed: z_at={cv_at_str} Error: {e}"

## Step 4. アンブレラサンプリングの並列実行
準備した関数 run_us_window を joblib を使って並列実行します。 例えば N_JOBS=4 ならば、4つのウィンドウ計算が同時に進行します。

In [ ]:
# ============================================================
# 4. Main Execution
# ============================================================

print(f"Start Parallel Calculation: {len(cv_restraint)} windows")
print(f"Settings: n_jobs={N_JOBS}, backend={BACKEND}")

# 並列実行
# delayed(関数名)(引数) for 変数 in リスト の形式で記述します
results = Parallel(n_jobs=N_JOBS, verbose=VERBOSE, backend=BACKEND)(
    delayed(run_us_window)(cv) for cv in cv_restraint
)

In [ ]:
# 結果の表示
print("\n--- Results ---")
for res in results:
    print(res)

## 補足

本ノートブックでは、アンブレラサンプリングによるデータ収集を行いました。 正確な自由エネルギー計算を行うためには、反応座標上において隣接するウィンドウ間のヒストグラムが十分に重なりがあることが重要です。もしオーバーラップが不足している場合、解析時に誤差が生じたり、計算が収束しない原因となります。オーバラップが十分でない場合は、バネ定数 `KAPPA` を弱くするか、その間に新しいウィンドウを追加する必要があります。

## Next Step

次の [06_mbar_free_energy_ja.ipynb](./06_mbar_free_energy_ja.ipynb) では、いよいよ最後の仕上げとして自由エネルギー計算の手順を見ていきます。